### Parte 1: Carregamento dos dados


In [2]:
# Importar a biblioteca pandas.
import pandas as pd
import numpy as np

In [3]:
# Carregando os dados a serem utilizados.
df_orcamento = pd.read_csv('Base_Orcamento.csv', sep=';', encoding='latin1')
df_realizado = pd.read_csv('Base_Realizado.csv', sep=';', encoding='latin1')


In [4]:
# Visualizar as primeiras linhas para ver se tudo ocorreu bem.
print("Orçamento:")
display(df_orcamento.head())
print("\nRealizado:")
display(df_realizado.head())

Orçamento:


,Obra,Grupo Orçamentário,Cód. Estruturado,Cód. Item,Descrição Item,Unid.,Categoria,Qtde. Insumo,Custo Insumo,Total Orçado,Tipo
0,OBR,Montagem de instalacoes provisorias - Impresso...,01010100101OBR,9832,Impressora multifuncional bivolt,PC,Materiais de Escritório e Informática,"1,00","538,03","538,03",Geral
1,OBR,Montagem de instalacoes provisorias - Conjunto...,01010100102OBR,8052,Cartucho compativel HP CARTRIDGE 667 preto,UN,Materiais de Escritório e Informática,"20,00","74,79","1.495,79",Geral
2,OBR,Montagem de instalacoes provisorias - Conjunto...,01010100102OBR,8050,Cartucho compativel HP CARTRIDGE 667 colorido,UN,Materiais de Escritório e Informática,"10,00","69,00","690,00",Geral
3,OBR,Limpeza de terreno - Servico terceirizado,01020100201OBR,10011,Servico preliminar de limpeza do terreno por area,M2,Serviço de Limpeza do Terreno,"2.628,24","8,17","21.459,57",Geral
4,OBR,Maquinario de escavacao de solo em obra - Area...,01020200201OBR,10225,Escavadeira locacao diaria,DIA,Locação de Equipamentos Pesados,"15,62","1.989,65","31.077,72",Geral



Realizado:


,Obra,Cód. Estruturado,Cód. Item,Descrição Item,Unid.,Categoria,Nº NF,Data NF,ValorUnitRealizado,Qtd Realizada,ValorTotalRealizado,Cód. Fornecedor,Fornecedor,Tipo,Cód. Pedido,Cód. Contrato
0,OBR,01010100102OBR,8052,Cartucho compativel HP CARTRIDGE 667 preto,UN,Materiais de Escritório e Informática,60634OBR.N,18/03/2025,"87,40","13,33","1.165,30",Código 01,Fornecedor 01,Geral,373OBR.P,-
1,OBR,01010100102OBR,8050,Cartucho compativel HP CARTRIDGE 667 colorido,UN,Materiais de Escritório e Informática,60634OBR.N,18/03/2025,"87,40","10,00","874,00",Código 01,Fornecedor 01,Geral,373OBR.P,-
2,OBR,01010100102OBR,8052,Cartucho compativel HP CARTRIDGE 667 preto,UN,Materiais de Escritório e Informática,60634OBR.N,18/03/2025,"92,00","6,67","613,36",Código 01,Fornecedor 01,Geral,373OBR.P,-
3,OBR,01010100102OBR,8050,Cartucho compativel HP CARTRIDGE 667 colorido,UN,Materiais de Escritório e Informática,60634OBR.N,18/03/2025,"92,00","5,00","460,00",Código 01,Fornecedor 01,Geral,373OBR.P,-
4,OBR,01010100102OBR,8052,Cartucho compativel HP CARTRIDGE 667 preto,UN,Materiais de Escritório e Informática,57638OBR.N,21/10/2024,"87,40","5,00","437,00",Código 01,Fornecedor 01,Geral,244OBR.P,-


### Parte 2: Limpeza dos dados


In [5]:
# --- Função para limpar e padronizar as duas tabelas ---
def limpar_dataframe(df):
    # 1. Padronizar nomes das colunas (minúsculas, sem espaços)
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    
    # 2. Identificar colunas de quantidade e valor
    cols_numericas = [col for col in df.columns if 'quantidade' in col or 'valor_unitario' in col]
    
    # 3. Corrigir o formato numérico (trocar ',' por '.' e converter para número)
    for col in cols_numericas:
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce') # 'coerce' transforma erros em NaN (Nulo)
        
    return df

# Aplicar a função de limpeza
df_orcamento = limpar_dataframe(df_orcamento.copy())
df_realizado = limpar_dataframe(df_realizado.copy())


In [6]:
# Remover colunas de valor_total originais, pois vamos calcular as nossas
df_orcamento = df_orcamento.drop(columns=['_total_orçado_'])
df_realizado = df_realizado.drop(columns=['_valortotalrealizado_'])

In [7]:
print(df_orcamento.columns)

Index(['obra', 'grupo_orçamentário', 'cód._estruturado', 'cód._item',
       'descrição_item', 'unid.', 'categoria', '_qtde._insumo_',
       '_custo_insumo_', 'tipo'],
      dtype='object')


### Passo 3: Criar as colunas calculadas.

In [8]:
# 1. Definir as colunas que precisam de limpeza em cada dataframe
cols_para_limpar_orc = ['_qtde._insumo_', '_custo_insumo_']
cols_para_limpar_real = ['_valorunitrealizado_', '_qtd_realizada_']

# 2. Limpar e converter as colunas para formato numérico
for col in cols_para_limpar_orc:
    # Remove o '.' de milhar, troca a ',' de decimal por '.' e converte para número
    df_orcamento[col] = pd.to_numeric(
        df_orcamento[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce' # Transforma erros de conversão em Nulo (NaN)
    )

for col in cols_para_limpar_real:
    df_realizado[col] = pd.to_numeric(
        df_realizado[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce'
    )

In [9]:

print("Colunas convertidas para número.")
df_orcamento['custo_total_orcado'] = df_orcamento['_qtde._insumo_'] * df_orcamento['_custo_insumo_']
df_realizado['custo_total_realizado'] = df_realizado['_qtd_realizada_'] * df_realizado['_valorunitrealizado_']
display(df_orcamento[['descrição_item', 'custo_total_orcado']].head(2))

Colunas convertidas para número.


,descrição_item,custo_total_orcado
0,Impressora multifuncional bivolt,538.03
1,Cartucho compativel HP CARTRIDGE 667 preto,1495.80


In [10]:
print(df_realizado.columns)

Index(['obra', 'cód._estruturado', 'cód._item', 'descrição_item', 'unid.',
       'categoria', 'nº_nf', 'data_nf', '_valorunitrealizado_',
       '_qtd_realizada_', 'cód._fornecedor', 'fornecedor', 'tipo',
       'cód._pedido', 'cód._contrato', 'custo_total_realizado'],
      dtype='object')


### Passo 4: Unir as Bases (Merge)

In [14]:
# Unir as duas tabelas usando a coluna 'cód._item' como a chave
# Usando um 'left' merge para manter todos os itens do orçamento.
df_final = pd.merge(
    df_orcamento,
    # Selecionando as colunas com os nomes exatos do seu df_realizado
    df_realizado[['cód._item', '_qtd_realizada_', '_valorunitrealizado_', 'custo_total_realizado']],
    on='cód._item',  # Usando a chave correta para a união
    how='left'
)

# Preencher com 0 os valores de itens orçados mas não realizados
# Usando os nomes de coluna corretos também nesta lista
cols_realizado_fill = ['_qtd_realizada_', '_valorunitrealizado_', 'custo_total_realizado']
df_final[cols_realizado_fill] = df_final[cols_realizado_fill].fillna(0)

display(df_final.head())

,obra,grupo_orçamentário,cód._estruturado,cód._item,descrição_item,unid.,categoria,_qtde._insumo_,_custo_insumo_,tipo,custo_total_orcado,_qtd_realizada_,_valorunitrealizado_,custo_total_realizado
0,OBR,Montagem de instalacoes provisorias - Impresso...,01010100101OBR,9832,Impressora multifuncional bivolt,PC,Materiais de Escritório e Informática,1.0,538.03,Geral,538.03,0.00,0.0,0.000
1,OBR,Montagem de instalacoes provisorias - Conjunto...,01010100102OBR,8052,Cartucho compativel HP CARTRIDGE 667 preto,UN,Materiais de Escritório e Informática,20.0,74.79,Geral,1495.80,13.33,87.4,1165.042
2,OBR,Montagem de instalacoes provisorias - Conjunto...,01010100102OBR,8052,Cartucho compativel HP CARTRIDGE 667 preto,UN,Materiais de Escritório e Informática,20.0,74.79,Geral,1495.80,6.67,92.0,613.640
3,OBR,Montagem de instalacoes provisorias - Conjunto...,01010100102OBR,8052,Cartucho compativel HP CARTRIDGE 667 preto,UN,Materiais de Escritório e Informática,20.0,74.79,Geral,1495.80,5.00,87.4,437.000
4,OBR,Montagem de instalacoes provisorias - Conjunto...,01010100102OBR,8052,Cartucho compativel HP CARTRIDGE 667 preto,UN,Materiais de Escritório e Informática,20.0,74.79,Geral,1495.80,6.67,87.4,582.958


### Passo 5: Análise de Variação

In [15]:
# Calcular as variações de custo
df_final['variacao_custo_abs'] = df_final['custo_total_realizado'] - df_final['custo_total_orcado']

# Calcular a variação percentual
df_final['variacao_custo_perc'] = np.where(
    df_final['custo_total_orcado'] != 0,
    df_final['variacao_custo_abs'] / df_final['custo_total_orcado'],
    0
)

display(df_final[['descrição_item', 'custo_total_orcado', 'custo_total_realizado', 'variacao_custo_abs', 'variacao_custo_perc']].head())

,descrição_item,custo_total_orcado,custo_total_realizado,variacao_custo_abs,variacao_custo_perc
0,Impressora multifuncional bivolt,538.03,0.000,-538.030,-1.000000
1,Cartucho compativel HP CARTRIDGE 667 preto,1495.80,1165.042,-330.758,-0.221124
2,Cartucho compativel HP CARTRIDGE 667 preto,1495.80,613.640,-882.160,-0.589758
3,Cartucho compativel HP CARTRIDGE 667 preto,1495.80,437.000,-1058.800,-0.707849
4,Cartucho compativel HP CARTRIDGE 667 preto,1495.80,582.958,-912.842,-0.610270


In [23]:

# Variação por PREÇO: (Preço Real - Preço Orçado) * Quantidade Real
df_final['variacao_por_preco'] = (df_final['_valorunitrealizado_'] - df_final['_custo_insumo_']) * df_final['_qtd_realizada_']

# Variação por QUANTIDADE: (Quantidade Real - Quantidade Orçada) * Preço Orçado
df_final['variacao_por_quantidade'] = (df_final['_qtd_realizada_'] - df_final['_qtde._insumo_']) * df_final['_custo_insumo_']

# Criando uma coluna de texto para classificar cada item
conditions = [
    (df_final['variacao_custo_perc'] > 0.10), # Estouro maior que 10%
    (df_final['variacao_custo_perc'] > 0),    # Estouro entre 0% e 10%
    (df_final['variacao_custo_perc'] == 0),   # Exatamente no orçamento
    (df_final['variacao_custo_perc'] < -0.10),# Economia maior que 10%
    (df_final['variacao_custo_perc'] < 0)     # Economia entre 0% e 10%
]

# Rótulos para cada condição
labels = ['Estouro Crítico', 'Estouro Moderado', 'No Orçamento', 'Ótima Economia', 'Pequena Economia']

# Aplicando a lógica à nova coluna
df_final['categoria_desvio'] = np.select(conditions, labels, default='Não Classificado')


# Verificando as novas colunas criadas
display(df_final[['descrição_item', 'variacao_por_preco', 'variacao_por_quantidade', 'categoria_desvio']].head())

# Exporanto resultado final
try:
    df_final.to_excel('Base_Analitica_Obra.xlsx', index=False)
    print("\nPROCESSO CONCLUÍDO COM SUCESSO!")
except Exception as e:
    print(f"\nOcorreu um erro ao salvar o arquivo Excel: {e}")

,descrição_item,variacao_por_preco,variacao_por_quantidade,categoria_desvio
0,Impressora multifuncional bivolt,-0.0000,-538.0300,Ótima Economia
1,Cartucho compativel HP CARTRIDGE 667 preto,168.0913,-498.8493,Ótima Economia
2,Cartucho compativel HP CARTRIDGE 667 preto,114.7907,-996.9507,Ótima Economia
3,Cartucho compativel HP CARTRIDGE 667 preto,63.0500,-1121.8500,Ótima Economia
4,Cartucho compativel HP CARTRIDGE 667 preto,84.1087,-996.9507,Ótima Economia



PROCESSO CONCLUÍDO COM SUCESSO!
